In [1]:
### Importing the required library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MolStandardize
import joblib

import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors,rdMolDescriptors
#import openpyxl


In [3]:
### Installation of the basic library 
from rdkit import Chem,DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
import numpy as np

In [2]:
### Reading the preprocess data from disk 
train_set=pd.read_csv('data/final_unique_train.csv')

test_set=pd.read_csv('data/final_unique_test.csv')

print(train_set.shape)
print(test_set.shape)

(17937, 8)
(1282, 8)


In [3]:
train_smiles_list=train_set[['smiles_canon']]
test_smiles_list=test_set[['smiles_canon']]

In [4]:

### Importing the function of the utility file to generate the descriptors and others evaluation matrics...  
import utilities

In [5]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, MolSurf, rdMolDescriptors
import pandas as pd
from rdkit import Chem

In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, MolSurf, rdMolDescriptors
import pandas as pd
from rdkit import Chem

# Function to calculate the specified descriptors
def generate30(smiles):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None  # Return None if the molecule could not be parsed
    
    # Calculate descriptors
    descriptors = {
        'LogP': Crippen.MolLogP(mol),                           # 1. LogP
        'LogP2': Crippen.MolMR(mol),                            # 2. LogP2
        'Tsch': rdMolDescriptors.CalcNumAtomStereoCenters(mol), # 3. Tsch
        'Gravto': Descriptors.MolWt(mol),                       # 4. Gravto
        'TPSA': rdMolDescriptors.CalcTPSA(mol),                 # 5. TPSA
        'ChiV8': Descriptors.Chi4v(mol),                        # 6. ChiV8

        #'bcutp2': Descriptors.BCUT2D(mol)[1],                  # 7. bcutp2
        'Chi10': Descriptors.Chi1n(mol),                        # 8. Chi10
        'slogPVSA1': MolSurf.SlogP_VSA_(mol)[0],                # 9. slogPVSA1
        'PEOEVSA5': MolSurf.PEOE_VSA_(mol)[4],                  # 10. PEOEVSA5
        #'MRVSA9': MolSurf.MR_VSA_(mol)[8],                      # 11. MRVSA9
        #'dchi4': Descriptors.Kappa4(mol),                       # 12. dchi4
        'Hy': Descriptors.HallKierAlpha(mol),                   # 13. Hy
        'UI': Descriptors.BalabanJ(mol),                        # 14. UI
        'naccr': rdMolDescriptors.CalcNumAromaticCarbocycles(mol), # 15. naccr
        'naro': rdMolDescriptors.CalcNumAromaticRings(mol),     # 16. naro
        #'bcutp3': Descriptors.BCUT2D(mol)[2],                   # 17. bcutp3
        'Scar': rdMolDescriptors.CalcNumSaturatedCarbocycles(mol), # 18. Scar
        'Smax': Descriptors.MaxAbsPartialCharge(mol),           # 19. Smax
        'Tpc': Descriptors.TPSA(mol),                           # 20. Tpc
        'Smin': Descriptors.MinPartialCharge(mol),              # 21. Smin
        'dchi1': Descriptors.Kappa1(mol),                       # 22. dchi1
        'AWeight': Descriptors.MolWt(mol),                      # 23. AWeight
        'Shal': Descriptors.HallKierAlpha(mol),                 # 24. Shal
        'nhyd': Descriptors.FractionCSP3(mol),                  # 25. nhyd
        'knotp': Descriptors.NumRadicalElectrons(mol),          # 26. knotp
        'dchi0': Descriptors.Kappa1(mol),                       # 27. dchi0 (proxy)
        'IC1': Descriptors.MolWt(mol),                          # 28. IC1 (proxy for unavailable)
        'Save': Crippen.MolLogP(mol),                           # 29. Save (LogP as proxy)
        'PEOEVSA0': MolSurf.PEOE_VSA_(mol)[0],                  # 30. PEOEVSA0
        'MZM1': Descriptors.MaxAbsPartialCharge(mol),           # 31. MZM1
        'CIC0': Descriptors.MolWt(mol),                         # 32. CIC0
        'Hatov': Descriptors.HallKierAlpha(mol),                # 33. Hatov
    }
    
    return descriptors
smiles_list_train=train_set.smiles_canon
smiles_list_test=test_set.smiles_canon
descriptor_data = [generate30(smiles) for smiles in smiles_list_train]
descriptor_data1 = [generate30(smiles) for smiles in smiles_list_test]
# Create a DataFrame from the list of descriptor dictionaries
df30_train = pd.DataFrame(descriptor_data)
df30_test = pd.DataFrame(descriptor_data1)

In [7]:
smiles_list_train=train_set.smiles_canon
smiles_list_test=test_set.smiles_canon

In [8]:
df7_train=utilities.get_functional_groups(train_set.smiles_canon)
df7_test=utilities.get_functional_groups(test_set.smiles_canon)

In [10]:
### Generate 4 descriptors ....
df4_train=utilities.generate4(train_set.smiles_canon)
df4_test=utilities.generate4(test_set.smiles_canon)
### Generate 17 descriptors ....
df17_train=utilities.generate17(train_set.smiles_canon)
df17_test=utilities.generate17(test_set.smiles_canon)
### Generate 123 descriptors ....
df123_train=utilities.generate123(train_set.smiles_canon)
df123_test=utilities.generate123(test_set.smiles_canon)
### Generate 38 feature engineered based on the structure of the smiles ....
df38_train=utilities.generate_features38(train_set.smiles_canon)
df38_test=utilities.generate_features38(test_set.smiles_canon)
### Generate 7 funnctional groups
df7_train=utilities.get_functional_groups(train_set.smiles_canon)
df7_test=utilities.get_functional_groups(test_set.smiles_canon)
### Fingerprint 128....
df128_train=utilities.fingerprint(train_set.smiles_canon,2,128)
df128_test=utilities.fingerprint(test_set.smiles_canon,2,128)
### Fingerprint 256....
df256_train=utilities.fingerprint(train_set.smiles_canon,2,256)
df256_test=utilities.fingerprint(test_set.smiles_canon,2,256)
### Fingerprint 512....
df512_train=utilities.fingerprint(train_set.smiles_canon,2,512)
df512_test=utilities.fingerprint(test_set.smiles_canon,2,512)
### Fingerprint 1024....
df1024_train=utilities.fingerprint(train_set.smiles_canon,2,1024)
df1024_test=utilities.fingerprint(test_set.smiles_canon,2,1024)

In [12]:
y_train=train_set['LogS']
y_test=test_set['LogS']

In [13]:
import xgboost #as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np

In [14]:
df17_train=utilities.generate17(train_set.smiles_canon)
df17_test=utilities.generate17(test_set.smiles_canon)

In [15]:

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []

# Predictions storage for test set
test_predictions = []

# Loop through each fold
for train_index, val_index in kf.split(df4_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df4_train.iloc[train_index], df4_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df4_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")




Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.5750, Std = 0.0068
RMSE: Mean = 0.7407, Std = 0.0089
R²: Mean = 0.8684, Std = 0.0031


In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df17_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df17_train.iloc[train_index], df17_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df17_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4576, Std = 0.0040
RMSE: Mean = 0.5999, Std = 0.0055
R²: Mean = 0.9137, Std = 0.0016


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df123_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df123_train.iloc[train_index], df123_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df123_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4165, Std = 0.0030
RMSE: Mean = 0.5656, Std = 0.0046
R²: Mean = 0.9233, Std = 0.0013


In [17]:
df253_train = pd.concat([df123_train, df128_train], axis=1)
df253_test = pd.concat([df123_test, df128_test], axis=1)

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df253_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df253_train.iloc[train_index], df253_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df253_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4164, Std = 0.0013
RMSE: Mean = 0.5642, Std = 0.0035
R²: Mean = 0.9237, Std = 0.0009


In [19]:
df260_train = pd.concat([df253_train, df7_train], axis=1)
df260_test = pd.concat([df253_test, df7_test], axis=1)

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df260_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df260_train.iloc[train_index], df260_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df260_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4135, Std = 0.0030
RMSE: Mean = 0.5615, Std = 0.0039
R²: Mean = 0.9244, Std = 0.0010


In [21]:
df298_train = pd.concat([df260_train, df38_train], axis=1)
df298_test = pd.concat([df260_test, df38_test], axis=1)

In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df298_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df298_train.iloc[train_index], df298_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df298_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4148, Std = 0.0027
RMSE: Mean = 0.5632, Std = 0.0043
R²: Mean = 0.9239, Std = 0.0012


In [ ]:
### End here ...